# Physics super-resolution with VAR (next-scale prediction)

Datasets: **CMS quark/gluon jet images** (`qg`, 3 x 125 x 125) and **CaloChallenge Dataset 2** (`calo`, 45 x 16 x 9 showers).

Pipeline, one cell per stage:

`download -> prepare -> tune_vqvae -> train_vqvae -> tune_var -> train_var -> generate -> eval_sr -> tune_tagger -> train_taggers -> summarize`

* Use a **GPU runtime** (Runtime -> Change runtime type -> T4 / L4 / A100).
* First run with `SMOKE = True` (tiny subset, 1-2 epochs) to check everything works, then set `SMOKE = False` and run all cells again.
* All results (checkpoints, metrics, Optuna studies, figures) go to Google Drive. Training and tuning resume from Drive after a disconnect, so after a disconnect just re-run the cells.
* Run one dataset per session: change `DATASET` and restart the runtime to switch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU - switch the runtime to GPU'
!df -h /content | tail -1

from google.colab import drive
drive.mount('/content/drive')

## 1. Get the code from GitHub

The first run clones the repository into `/content/superres_code`. Every re-run pulls the latest commit of `BRANCH`, so code pushed to GitHub is picked up without restarting the runtime.

In [ ]:
GIT_URL = 'https://github.com/rajveer43/physics-superres-var.git'
BRANCH  = 'main'
CODE    = '/content/superres_code'

import os, shutil, subprocess, sys

def git(*args, cwd=CODE):
    r = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(CODE, '.git')):
    git('fetch', '--depth', '1', 'origin', BRANCH)
    git('reset', '--hard', f'origin/{BRANCH}')   # pull; discards any edits made inside Colab
else:
    shutil.rmtree(CODE, ignore_errors=True)
    git('clone', '--depth', '1', '--branch', BRANCH, GIT_URL, CODE, cwd='/content')
print('code at commit:', git('log', '-1', '--format=%h %s (%cr)'))

if CODE not in sys.path:
    sys.path.insert(0, CODE)
for m in [m for m in sys.modules if m.split('.')[0] in ('superres', 'tests')]:
    del sys.modules[m]   # re-import the freshly pulled code

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, pandas, matplotlib, h5py, pyarrow, requests
!pip install -q optuna tabulate
import torch, optuna
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| optuna', optuna.__version__)

## 2. Settings

Every value in `superres/config.py` can be overridden with a dotted key in `OVERRIDES`, e.g. `{'var.epochs': 20, 'data.max_samples': 20000, 'tagger.seeds': [42, 43, 44]}`.

Smoke runs use separate cache and Drive folders (`*_smoke`), so they never mix with the real results.

In [ ]:
DATASET    = 'qg'     # 'qg' (CMS quark/gluon jets) or 'calo' (CaloChallenge DS2)
SMOKE      = True     # tiny subset + 1-2 epochs: checks the whole pipeline quickly
FAKE_DATA  = False    # synthetic data instead of the download (fastest check; implies SMOKE)
RUN_TUNING = True     # False: skip Optuna, use config defaults or an earlier *__best.json
LEVELS     = None     # None = all LR levels, or e.g. ['pool4x4']
SEEDS      = None     # tagger seeds; None = config value. Use e.g. [42, 43, 44] for error bars
OVERRIDES  = {}

import json, time
import pandas as pd
from superres import pipeline
from superres.config import get_config
from superres.io_utils import RunDir

SMOKE = SMOKE or FAKE_DATA
tag = '_fake' if FAKE_DATA else ('_smoke' if SMOKE else '')
paths = {
    'paths.drive_root': f'/content/drive/MyDrive/superres_results{tag}',
    'paths.raw_root':   '/content/data/raw_fake' if FAKE_DATA else '/content/data/raw',
    'paths.cache_root': f'/content/data/cache{tag}',
}
cfg = get_config(DATASET, {**paths, **OVERRIDES}, smoke=SMOKE)

def stage(name, **kw):
    t0 = time.time()
    print(f'######## {DATASET}: {name} ########')
    out = pipeline.run_stage(cfg, name, levels=LEVELS, seeds=SEEDS, **kw)
    print(f'-- {name} finished in {(time.time() - t0) / 60:.1f} min')
    return out

def history(stage_name, variant, seed=cfg['seed'], n=5):
    path = RunDir(cfg, stage_name, variant, seed).file('history.csv')
    return pd.read_csv(path).tail(n) if os.path.exists(path) else 'no history yet'

print('dataset :', DATASET, '| smoke:', SMOKE, '| fake data:', FAKE_DATA)
print('levels  :', cfg['data']['levels'])
print(json.dumps(cfg['paths'], indent=1))

## 3. Download

* `calo`: `dataset_2_1.hdf5` (train/val) and `dataset_2_2.hdf5` (test) from Zenodo record 6366271, about 1.4 GB each, md5-checked.
* `qg`: every file in the public CERNBox share. The share is listed over WebDAV and each file fetched with `wget`. If that fails, archive download of the whole share is tried. If both fail, paste direct file links into `OVERRIDES = {'data.extra_urls': [...]}` or copy the files into `/content/data/raw/qg/` yourself.

Downloads resume (`wget -c`), and files that are already complete are skipped.

In [ ]:
if FAKE_DATA:
    from tests.smoke_test import fake_calo, fake_qg
    raw = os.path.join(cfg['paths']['raw_root'], DATASET)
    if not (os.path.isdir(raw) and os.listdir(raw)):
        (fake_qg if DATASET == 'qg' else fake_calo)(raw)
    print(os.listdir(raw))
else:
    stage('download')

## 4. Prepare: energy-preserving down-sampling

Writes numpy memmaps to local disk: HR ground truth (zero-padded; qg 125 -> 128, calo 45 -> 48 layers) and each LR level made by **summing** blocks of cells, so every coarse cell holds exactly the energy of the fine cells it covers.

Check the printed label counts (qg) and `negative_fraction_clipped`. The step is skipped if the cache already exists; delete `/content/data/cache*/<dataset>` to rebuild it.

In [ ]:
meta = stage('prepare')
print(json.dumps({k: v for k, v in meta.items() if k != 'source_files'}, indent=1))
!du -sh {cfg['paths']['cache_root']}/{DATASET}

## 5. Multi-scale VQ-VAE (tokenizer)

The Optuna study tunes the learning rate, codebook size, latent channels, beta, energy-loss weight and width. Its objective is validation log-energy MSE + median |dE/E|. Full training then uses the best parameters automatically.

In [ ]:
if RUN_TUNING:
    stage('tune_vqvae')

In [ ]:
stage('train_vqvae')
history('vqvae', 'hr')

## 6. Conditional VAR transformer (one model per LR level)

Tuned once on the hardest level (depth with width = 64 x depth as in the paper, lr, dropout, weight decay, label smoothing; objective: validation token cross-entropy). The best parameters are reused for every level.

In [ ]:
if RUN_TUNING:
    stage('tune_var')

In [ ]:
stage('train_var')
for lvl in (LEVELS or cfg['data']['levels']):
    print(lvl); display(history('var', lvl))

## 7. Generate SR for train / val / test

The VAR output for every event is cached on local disk. The taggers are trained on it, so all splits are generated.

In [ ]:
stage('generate')
!du -sh {cfg['paths']['cache_root']}/{DATASET}

## 8. SR quality in physics terms

For every level, the LR input, uniform upsampling, raw VAR and energy-projected VAR are compared with HR on physics observables (jets: pT, mass, girth, pT_D, tau21, hit multiplicity, radial profile; showers: E_dep / E_inc, shower depth and width, lateral centroid and width, energy fractions by depth, profiles). The comparison uses:

* **W1 / sigma_HR**: Wasserstein distance in units of the true spread (0 = identical distributions)
* **per-event bias and resolution** of the observable against the true event
* **classifier two-sample test AUC** (0.5 = a CNN cannot tell SR from HR)
* **LR closure**: how exactly SR reproduces the measured coarse cells

In [ ]:
sr_tables = stage('eval_sr')
lvl = (LEVELS or list(cfg['data']['levels']))[-1]
sr_tables[lvl].pivot_table(index='observable', columns='method', values='w1_over_sigma').round(3)

## 9. Downstream taggers

The architecture is tuned once on HR and the same one is trained separately on every input (HR, each LR grid, uniform upsampling, VAR). Any difference in results therefore comes from the input data, not from how much tuning each model got.

* `qg`: quark vs gluon classifier. Reports ROC AUC and background rejection 1/eps_B at signal efficiency 30/50/70%, with bootstrap uncertainties.
* `calo`: incident-energy regressor. Reports response bias and resolution sigma_eff/E, overall and in energy bins.

In [ ]:
if RUN_TUNING:
    stage('tune_tagger')

In [ ]:
stage('train_taggers')

## 10. Summary tables and figures (saved to Drive under `<dataset>/summary/`)

In [ ]:
import glob
from IPython.display import Image, Markdown, display
from superres import io_utils

stage('summarize')
sdir = io_utils.summary_dir(cfg)
display(Markdown(open(os.path.join(sdir, f'{DATASET}__report.md')).read()))
for f in sorted(glob.glob(os.path.join(sdir, 'figures', '*.png'))):
    print(os.path.basename(f)); display(Image(f, width=560))

In [ ]:
# Per-level figures: observable histograms with ratio panels, mean profiles, example events
lvl = (LEVELS or list(cfg['data']['levels']))[-1]
fig_dir = RunDir(cfg, 'sreval', lvl).fig_dir
for f in sorted(glob.glob(os.path.join(fig_dir, '*.png')))[:12]:
    print(os.path.basename(f)); display(Image(f, width=420))

## 11. Housekeeping

The local cache is lost when the runtime is recycled; results on Drive are kept. To switch datasets in the same runtime, free the disk first.

In [ ]:
# !rm -rf /content/data/cache/{DATASET} /content/data/raw/{DATASET}
!df -h /content | tail -1
!du -sh /content/data/* 2>/dev/null